In [1]:
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration

model_id = "llava-hf/llava-1.5-7b-hf"

processor = AutoProcessor.from_pretrained(model_id)
print("Processor loaded successfully!")

model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)
print("Model loaded successfully!")

/venv/vlm-llava/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Processor loaded successfully!


Loading weights: 100%|██████████| 686/686 [00:02<00:00, 236.78it/s, Materializing param=model.vision_tower.vision_model.pre_layrnorm.weight]                        


Model loaded successfully!


In [2]:
# ============ COCO VAL2017 ============
COCO_IMG_DIR = "../data/coco2017/val2017"
COCO_ANN_PATH = "../data/coco2017/annotations/instances_val2017.json"
OUTPUT_PATH = "../results/infer/coco_val2017/llava15/normal.json"
BATCH_SIZE = 16
MAX_NEW_TOKENS = 256

In [3]:
import json
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import torch

PROMPT = "Describe this image."

processor.tokenizer.padding_side = "left"
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token


def batch_list(items, batch_size):
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]


def load_image(path):
    return Image.open(path).convert("RGB")


def clean_output(text):
    text = text.strip()
    if "ASSISTANT:" in text:
        text = text.split("ASSISTANT:", 1)[-1].strip()
    return text


# -----------------------------
# Load COCO val2017 annotation
# -----------------------------
with open(COCO_ANN_PATH, "r", encoding="utf-8") as f:
    coco = json.load(f)

images_info = sorted(coco["images"], key=lambda x: x["id"])

print(f"Loaded {len(images_info)} COCO val2017 images.")
print("Example:", images_info[0])


# -----------------------------
# Run inference
# -----------------------------
results = []

hf_prompt = f"USER: <image>\n{PROMPT} ASSISTANT:"

for batch in tqdm(list(batch_list(images_info, BATCH_SIZE))):
    batch_images = []
    batch_prompts = []
    batch_ids = []

    for item in batch:
        image_id = int(item["id"])
        file_name = item["file_name"]

        image_path = Path(COCO_IMG_DIR) / file_name

        if not image_path.exists():
            raise FileNotFoundError(f"Missing image: {image_path}")

        image = load_image(image_path)

        batch_images.append(image)
        batch_prompts.append(hf_prompt)
        batch_ids.append(image_id)

    inputs = processor(
        text=batch_prompts,
        images=batch_images,
        return_tensors="pt",
        padding=True
    )

    input_device = next(model.parameters()).device
    inputs = {
        k: v.to(input_device) if isinstance(v, torch.Tensor) else v
        for k, v in inputs.items()
    }

    input_len = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=processor.tokenizer.eos_token_id
        )

    generated_ids = output_ids[:, input_len:]

    outputs = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

    for image_id, output in zip(batch_ids, outputs):
        caption = clean_output(output)

        results.append({
            "id": image_id,
            "caption": caption
        })

# -----------------------------
# Save id-caption only
# -----------------------------
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"\nSaved {len(results)} captions to {OUTPUT_PATH}")

Loaded 5000 COCO val2017 images.
Example: {'license': 2, 'file_name': '000000000139.jpg', 'coco_url': 'http://images.cocodataset.org/val2017/000000000139.jpg', 'height': 426, 'width': 640, 'date_captured': '2013-11-21 01:34:01', 'flickr_url': 'http://farm9.staticflickr.com/8035/8024364858_9c41dc1666_z.jpg', 'id': 139}


100%|██████████| 313/313 [54:49<00:00, 10.51s/it]


Saved 5000 captions to ../results/infer/coco_val2017/llava15/normal.json
